# Quickstart

This notebook walks you through a complete `f3dasm` workflow: defining a parameter space, sampling experiments, evaluating them with a benchmark function, and running an optimization loop.

By the end, you'll understand the core building blocks and how they fit together.

## 1. Define a parameter space

Every experiment starts with a **Domain** — the parameter space that defines what inputs your experiment can have.

In [ ]:
from f3dasm.design import Domain

domain = Domain()
domain.add_float(name="x0", low=-5.0, high=5.0)
domain.add_float(name="x1", low=-5.0, high=5.0)

domain

## 2. Create an ExperimentData object

**ExperimentData** is the central data container that holds all your experiments (inputs and outputs).

In [ ]:
from f3dasm import ExperimentData

data = ExperimentData(domain=domain)
data

## 3. Sample initial experiments

Use a **sampler** to generate initial experiments. Here we use Latin Hypercube sampling to create 10 samples.

In [ ]:
from f3dasm import create_sampler

sampler = create_sampler("latin", seed=42)
data = sampler.call(data, n_samples=10)

data

## 4. Evaluate experiments

Use a **data generator** to evaluate each sample. Here we use the built-in Sphere function ($f(x) = \sum x_i^2$) as our objective.

In [ ]:
from f3dasm import create_datagenerator

evaluator = create_datagenerator(
    "sphere", output_names="y", scale_bounds=[[-5.0, 5.0], [-5.0, 5.0]]
)

evaluator.arm(data)
data = evaluator.call(data)

data

## 5. Inspect the results

You can convert the data to a pandas DataFrame for inspection.

In [ ]:
df_input, df_output = data.to_pandas()

print("Best result:")
best_idx = df_output["y"].idxmin()
print(f"  x0 = {df_input.loc[best_idx, 'x0']:.4f}")
print(f"  x1 = {df_input.loc[best_idx, 'x1']:.4f}")
print(f"  y  = {df_output.loc[best_idx, 'y']:.4f}")

## 6. Create a custom Block

You can create your own computation blocks using the `@datagenerator` decorator:

In [ ]:
from f3dasm import datagenerator


@datagenerator(output_names=["y"])
def my_function(x0: float, x1: float) -> float:
    """A simple custom objective function."""
    return (x0 - 1) ** 2 + (x1 + 2) ** 2


# Use it like any other data generator
data2 = ExperimentData(domain=domain)
data2 = sampler.call(data2, n_samples=10)
my_function.arm(data2)
data2 = my_function.call(data2)

data2

## Next steps

Now that you've seen the basics, explore the tutorials in more depth:

- [Creating a Domain](design/domain_creation.ipynb) — all parameter types and advanced domain features
- [Working with ExperimentData](experimentdata/experimentdata.ipynb) — storage, export, and data manipulation
- [Understanding Blocks](data-driven/blocks.ipynb) — custom blocks, the Block class, and the decorator pattern
- [Building a Pipeline](pipeline/pipeline.ipynb) — chain blocks into automated workflows
- [Built-in Defaults](../defaults.md) — all available samplers, benchmark functions, and optimizers